In [3]:
import spacy
from spacy import displacy
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification
import json
import re

# Load model
tokenizer = AutoTokenizer.from_pretrained("cahya/bert-base-indonesian-NER")
model = AutoModelForTokenClassification.from_pretrained("cahya/bert-base-indonesian-NER")
ner_pipeline = pipeline("ner", model=model, tokenizer=tokenizer,
                        aggregation_strategy="simple")

def clean_text(text):
    # Hapus pola nomor halaman + judul buku: "2 P a n j i s e t a m a n"
    # Polanya: angka lalu huruf-huruf yang dipisah spasi
    text = re.sub(r'\d+\s+(?:[A-Za-z]\s+){2,}[A-Za-z]+', '', text)
    
    # Hapus pola "Kumpulan Dongeng nusantara 9" (judul + nomor halaman)
    text = re.sub(r'Kumpulan Dongeng nusantara\s+\d+', '', text)
    
    # Hapus baris yang hanya berisi angka (nomor halaman standalone)
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)
    
    # Hapus baris pendek yang kemungkinan besar header/footer (< 4 kata)
    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        words = line.strip().split()
        # Simpan baris kalau lebih dari 3 kata ATAU kosong (jaga struktur paragraf)
        if len(words) > 3 or len(words) == 0:
            cleaned_lines.append(line)
    text = '\n'.join(cleaned_lines)
    
    # Rapikan spasi berlebih
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)
    
    return text.strip()

# Fungsi chunking
def chunk_text(text, max_length=400):
    import re
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks, current = [], ""
    for sent in sentences:
        if len(current) + len(sent) < max_length:
            current += " " + sent
        else:
            if current:
                chunks.append(current.strip())
            current = sent
    if current:
        chunks.append(current.strip())
    return chunks

# Fungsi konversi ke format displaCy
def to_displacy_format(text, hf_entities):
    ents = []
    for ent in hf_entities:
        ents.append({
            "start": ent["start"],
            "end": ent["end"],
            "label": ent["entity_group"]  # hasil aggregation_strategy="simple"
        })
    
    return {
        "text": text,
        "ents": ents,
        "title": None
    }

# Load dataset
with open(r"C:\Semester 4\NLP\Pembelajaran Bahasa Mendalam\Tugas NER\data\dataset_ner_200.json", encoding="utf-8") as f:
    data = json.load(f)

# Ambil 1 dokumen dulu untuk dicoba
teks = data[0]["isi"]
teks = clean_text(teks)       # <-- tambahkan ini
chunks = chunk_text(teks)

# Proses tiap chunk dan visualisasikan
for i, chunk in enumerate(chunks[:5]):  # batasi 3 chunk dulu biar tidak terlalu panjang
    entities = ner_pipeline(chunk)
    
    doc_data = to_displacy_format(chunk, entities)
    
    print(f"\n--- Chunk {i+1} ---")
    displacy.render(doc_data, style="ent", manual=True, jupyter=True)

Some weights of the model checkpoint at cahya/bert-base-indonesian-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.



--- Chunk 1 ---



--- Chunk 2 ---



--- Chunk 3 ---



--- Chunk 4 ---



--- Chunk 5 ---


In [1]:
import spacy
from spacy import displacy
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification
import json
import re
import pandas as pd

# ============================================================
# 1. LOAD MODEL
# ============================================================
tokenizer = AutoTokenizer.from_pretrained("cahya/bert-base-indonesian-NER")
model = AutoModelForTokenClassification.from_pretrained("cahya/bert-base-indonesian-NER")

# Pipeline untuk displaCy (aggregated)
ner_pipeline = pipeline("ner", model=model, tokenizer=tokenizer,
                        aggregation_strategy="simple")

# Pipeline untuk IOB tags (raw)
ner_pipeline_raw = pipeline("ner", model=model, tokenizer=tokenizer)

# ============================================================
# 2. FUNGSI PREPROCESSING
# ============================================================
def clean_text(text):
    text = re.sub(r'\d+\s+(?:[A-Za-z]\s+){2,}[A-Za-z]+', '', text)
    text = re.sub(r'Kumpulan Dongeng nusantara\s+\d+', '', text)
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)
    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        words = line.strip().split()
        if len(words) > 3 or len(words) == 0:
            cleaned_lines.append(line)
    text = '\n'.join(cleaned_lines)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

def chunk_text(text, max_length=400):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks, current = [], ""
    for sent in sentences:
        if len(current) + len(sent) < max_length:
            current += " " + sent
        else:
            if current:
                chunks.append(current.strip())
            current = sent
    if current:
        chunks.append(current.strip())
    return chunks

# ============================================================
# 3. FUNGSI KONVERSI KE FORMAT DISPLACY
# ============================================================
def to_displacy_format(text, hf_entities):
    ents = []
    for ent in hf_entities:
        ents.append({
            "start": ent["start"],
            "end":   ent["end"],
            "label": ent["entity_group"]
        })
    return {"text": text, "ents": ents, "title": None}

# ============================================================
# 4. LOAD DATASET
# ============================================================
with open(r"C:\Semester 4\NLP\Pembelajaran Bahasa Mendalam\Tugas NER\data\dataset_ner_200.json", encoding="utf-8") as f:
    data = json.load(f)

# ============================================================
# 5. PROSES SEMUA DOKUMEN (agregasi + IOB)
# ============================================================
all_aggregated = []   # untuk displaCy dan rekap entitas
all_iob_rows   = []   # untuk tabel IOB tags

for doc_idx, item in enumerate(data):
    teks = clean_text(item["isi"])
    chunks = chunk_text(teks)

    for chunk in chunks:
        # --- Aggregated (untuk displaCy & rekap) ---
        agg_entities = ner_pipeline(chunk)
        all_aggregated.append({
            "doc_idx": doc_idx,
            "chunk":   chunk,
            "entities": agg_entities
        })

        # --- Raw IOB (untuk tabel) ---
        raw_entities = ner_pipeline_raw(chunk)
        for ent in raw_entities:
            if ent["entity"] != "O":   # buang token non-entitas
                all_iob_rows.append({
                    "doc_idx": doc_idx,
                    "word":    ent["word"],
                    "iob_tag": ent["entity"],
                    "score":   round(ent["score"], 4)
                })

print(f"Total chunks diproses: {len(all_aggregated)}")

# ============================================================
# 6. VISUALISASI DISPLACY (per chunk, ambil dokumen pertama)
# ============================================================
colors = {
    "PER": "#f9a825",
    "LOC": "#42a5f5",
    "ORG": "#66bb6a",
}

print("\n====== VISUALISASI NER - DOKUMEN 0 ======")
doc0_chunks = [x for x in all_aggregated if x["doc_idx"] == 0]

for i, item in enumerate(doc0_chunks):
    doc_data = to_displacy_format(item["chunk"], item["entities"])
    print(f"\n--- Chunk {i+1} ---")
    displacy.render(doc_data, style="ent", manual=True, jupyter=True,
                    options={"colors": colors})

# ============================================================
# 7. TABEL IOB TAGS (dokumen pertama)
# ============================================================
df_iob = pd.DataFrame(all_iob_rows)
df_iob_doc0 = df_iob[df_iob["doc_idx"] == 0].reset_index(drop=True)

print("\n====== IOB TAGS - DOKUMEN 0 ======")
display(df_iob_doc0.head(30))

# ============================================================
# 8. REKAP SEMUA ENTITAS YANG DITEMUKAN
# ============================================================
rekap = []
for item in all_aggregated:
    for ent in item["entities"]:
        rekap.append({
            "doc_idx":      item["doc_idx"],
            "entity":       ent["word"],
            "label":        ent["entity_group"],
            "score":        round(ent["score"], 4)
        })

df_rekap = pd.DataFrame(rekap)
print("\n====== REKAP SEMUA ENTITAS ======")
display(df_rekap.head(30))

# Frekuensi entitas per label
print("\n====== FREKUENSI PER LABEL ======")
display(df_rekap["label"].value_counts())

# Entitas paling sering muncul
print("\n====== ENTITAS PALING SERING MUNCUL ======")
display(df_rekap.groupby(["entity", "label"]).size()
        .reset_index(name="frekuensi")
        .sort_values("frekuensi", ascending=False)
        .head(20))

Some weights of the model checkpoint at cahya/bert-base-indonesian-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu
Device set to use cpu
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


: 